# Giai đoạn 1c: Đặc trưng phi tuyến có thắng được ridge tốt nhất không? (chỉ cần CPU)

**Vị trí trong lộ trình.** Mục tiêu cuối là một model encoder–decoder SOTA cho bài toán EEG → iEEG bipolar. Các giai đoạn 0, 1a, 1b là bước EDA: đo trần thông tin và kiểm tra từng đòn bẩy bằng model rẻ trước khi dùng GPU. Giai đoạn 1c là **cổng quyết định** có nên xây encoder phi tuyến hay không, và nếu xây thì encoder nên trích kiểu đặc trưng nào.

Giai đoạn 1b cho thấy:
- `ridge_kfold_shared` là baseline mạnh mới (+0,008, 15/15 subject).
- Envelope có thông tin ở dải alpha/beta, nhưng tầng 2 chọn alpha theo r nên cộng thêm nhiễu (R² = −0,16).

Ở giai đoạn này, tầng 2 được fit trên **phần dư out-of-fold** của baseline, **chọn alpha theo MSE**, và có **cổng theo từng contact**: chỉ bật khi MSE trên CV thắng việc dự đoán bằng trung bình. Nhờ vậy tầng 2 chỉ cộng thêm những gì cross-validation xác nhận là thật.

| Model | Tầng 2 (có cổng) dùng | Câu hỏi trả lời |
|---|---|---|
| `ridge` | (không có) | Baseline cũ của `train_loso.py` |
| `base` | (không có) | `ridge_kfold_shared` của giai đoạn 1b, là baseline cần thắng |
| `env_gated` | Waveform alpha/beta + envelope theta/alpha/beta | Sửa `stack_env` của 1b |
| `wav_gated` | Envelope Morlet theo từng kênh, 12 dải 2–64 Hz | Công suất phổ theo kênh (gồm cả gamma) có giúp không? |
| `rocket_gated` | 512 đặc trưng ngẫu nhiên: bộ lọc không gian → wavelet Morlet ngẫu nhiên → envelope/ReLU | Đặc trưng phi tuyến tổng quát, kể cả tương tác giữa các kênh, có giúp không? |
| `all_gated` | `wav` + `rocket` | Gộp lại có thêm gì không? |

Tập test cố định (các trial sau mốc 20%) và lượng calib 5/10/20% giống hệt giai đoạn 0, 1a, 1b, nên so sánh trực tiếp được với các giai đoạn trước.

- **Runtime: CPU** (Runtime → Change runtime type → CPU).
- Chạy lại được nhiều lần: subject nào đã xong sẽ được bỏ qua.

In [ ]:
#@title 1. Mount Drive + đường dẫn
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
DATA_DIR = f'{ROOT}/data/s2i_v1'
OUT_DIR  = f'{ROOT}/runs/nonlin_bip_car'
os.makedirs(OUT_DIR, exist_ok=True)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
for f in ('ceiling.py', 'within.py'):
    assert os.path.exists(f'{CODE_DIR}/s2i/{f}'), f'Thiếu s2i/{f}: cần chạy notebook giai đoạn 0 và 1b trước'
print('subjects:', len(glob.glob(f'{DATA_DIR}/sub-*.pt')))

## 2. Ghi `s2i/nonlin.py` lên Drive

In [ ]:
%%writefile {CODE_DIR}/s2i/nonlin.py
"""
Phase 1c (CPU) - can ANY non-linear EEG feature beat a well-regularised linear model with a
few minutes of calibration?

Phase 1b: the only linear lever that helped was better regularisation (ridge with one alpha
shared by all contacts, chosen by K-fold: +0.008 at 1-2 min, 15/15 subjects). Envelope
features raised alpha-band r by 17 % but the residual stage was selected by Pearson r, which
does not penalise amplitude, so it added noise (R^2 -> -0.16) and hurt small subjects.

Here the second stage is fitted on the OUT-OF-FOLD residual of the linear model and is
  * selected by MSE (not r), per contact, and
  * GATED: switched on for a contact only if its cross-validated MSE beats predicting the
    residual mean.
so it can only add what cross-validation says is real.

Models
------
  ridge         baseline of train_loso.py / ceiling.py (per-contact alpha, one 80/20 split)
  base          ridge_kfold_shared from Phase 1b (new strong baseline)
  env_gated     base + gated stage 2 on alpha/beta band waveforms + theta/alpha/beta envelopes
  wav_gated     base + gated stage 2 on per-channel Morlet envelopes, 12 log-spaced bands 2-64 Hz
                (exhaustive per-channel spectral power, incl. gamma)
  rocket_gated  base + gated stage 2 on random non-linear convolutional features
                (random spatial filter -> random Morlet wavelet -> envelope or ReLU).
                A cheap stand-in for "a non-linear encoder", with no training.
  all_gated     base + gated stage 2 on [wavelet envelopes, random features]

Usage
-----
python -m s2i.nonlin --data DATA_DIR --out RUNS_DIR/nonlin_bip_car
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
from scipy.fft import next_fast_len
from scipy.signal import butter, hilbert, sosfiltfilt

from s2i.baselines import lagged
from s2i.ceiling import StreamRidge, chrono_split
from s2i.metrics import per_contact_metrics
from s2i.within import ALPHAS_WIDE, LAG9, KFoldRidge, _blocks

FS = 200
ENV_LAGS = (-20, -10, 0, 10, 20)
_B = {k: butter(4, f, btype="band", fs=FS, output="sos")
      for k, f in {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30)}.items()}


# ----------------------------------------------------------------------------- features
def feat_env(e):
    a = sosfiltfilt(_B["alpha"], e, axis=1)
    b = sosfiltfilt(_B["beta"], e, axis=1)
    t = sosfiltfilt(_B["theta"], e, axis=1)
    parts = [a, b, np.abs(hilbert(t, axis=1)), np.abs(hilbert(a, axis=1)), np.abs(hilbert(b, axis=1))]
    return lagged(np.concatenate(parts, 0).astype(np.float32), ENV_LAGS)


def morlet(freq, cycles):
    """Complex Morlet kernels (len(freq), 2H+1), unit L1 norm, centred at index H."""
    sig = np.asarray(cycles) / (2 * np.pi * np.asarray(freq))        # Gaussian width, s
    H = int(np.ceil(3 * sig.max() * FS))
    t = np.arange(-H, H + 1) / FS
    w = np.exp(-t[None] ** 2 / (2 * sig[:, None] ** 2)) * np.exp(2j * np.pi * np.asarray(freq)[:, None] * t[None])
    return (w / np.abs(w).sum(1, keepdims=True)).astype(np.complex64), H


def wconv(S, kern, H):
    """Zero-phase convolution of each row of S (M, L) with its own kernel (M, 2H+1) -> complex (M, L)."""
    L = S.shape[1]
    n = next_fast_len(L + 2 * H)
    return np.fft.ifft(np.fft.fft(S, n) * np.fft.fft(kern, n), n)[:, H:H + L]


class MorletEnvBank:
    """Per-channel Morlet envelope at n_freq log-spaced frequencies (all channels x all bands), lagged."""

    def __init__(self, freqs=tuple(np.geomspace(2, 64, 12)), cycles=5, lags=(-20, 0, 20)):
        self.kern1, self.H = morlet(freqs, np.full(len(freqs), cycles))
        self.nf, self.lags = len(freqs), lags

    def __call__(self, e):
        C = e.shape[0]
        S = np.repeat(e, self.nf, 0)
        env = np.abs(wconv(S, np.tile(self.kern1, (C, 1)), self.H))
        return lagged(env.astype(np.float32), self.lags)


class RandomConvFeatures:
    """
    M random (spatial filter -> complex Morlet wavelet -> non-linearity) features, fixed seed.
    A cheap stand-in for the first layer of a convolutional encoder, with no training:
      spatial filter : 1/3 single channel, 1/3 channel-pair difference, 1/3 dense random
      wavelet        : centre frequency log-uniform in [fmin, fmax] Hz, 3-7 cycles
      non-linearity  : half envelope |w * s|, half ReLU(Re(w * s) - b * rms), b ~ U(-1, 1)
    mode (same random filters/wavelets whatever the mode, so modes form a matched ablation):
      "mix" (above) | "env" all envelopes | "relu" all ReLU | "linear" Re(w * s), no non-linearity
    Synthetic check: dense spatial filters + short random 9-tap kernels (plain ROCKET) missed an
    alpha-envelope signal that feat_env recovered - broadband kernels dilute narrow-band power.
    """

    def __init__(self, n_chan, M=512, fmin=2.0, fmax=60.0, seed=0, mode="mix"):
        assert mode in ("mix", "env", "relu", "linear")
        self.mode = mode
        g = np.random.default_rng(seed)
        self.Ws = np.zeros((M, n_chan), np.float32)
        kind = np.arange(M) % 3
        for i in range(M):
            if kind[i] == 0 or n_chan < 2:
                self.Ws[i, g.integers(n_chan)] = 1
            elif kind[i] == 1:
                c = g.choice(n_chan, 2, replace=False)
                self.Ws[i, c] = (1, -1)
            else:
                self.Ws[i] = g.normal(size=n_chan)
        self.Ws /= np.linalg.norm(self.Ws, axis=1, keepdims=True)
        self.freq = np.exp(g.uniform(np.log(fmin), np.log(fmax), M))
        self.kern, self.H = morlet(self.freq, g.uniform(3, 7, M))
        self.env = g.random(M) < 0.5
        self.bias = g.uniform(-1, 1, M).astype(np.float32)

    def __call__(self, e):
        C = wconv(self.Ws @ e, self.kern, self.H)
        R = C.real
        if self.mode == "linear":
            return R.astype(np.float32)
        if self.mode == "env":
            return np.abs(C).astype(np.float32)
        out = np.maximum(R - self.bias[:, None] * np.sqrt((R * R).mean(1, keepdims=True)), 0)
        if self.mode == "mix":
            out[self.env] = np.abs(C[self.env])
        return out.astype(np.float32)


def concat_feats(*fs):
    return lambda e: np.concatenate([f(e) for f in fs], 0)


# ----------------------------------------------------------------------------- gated stage 2
class GatedResidualRidge:
    """
    Ridge on a residual target with per-contact alpha chosen by K-fold MSE, and a per-contact
    gate: ON only if the best CV MSE beats predicting the training-fold mean by `margin`.
    """

    def __init__(self, feat, alphas=ALPHAS_WIDE, k=5, margin=0.002):
        self.feat, self.alphas, self.k, self.margin = feat, alphas, k, margin
        self._sr = StreamRidge(feat, alphas)

    def fit(self, E, R):
        n, K = len(E), R[0].shape[0]
        folds = _blocks(n, self.k)
        fold_of = np.empty(n, int)
        for fi, idx in enumerate(folds):
            fold_of[idx] = fi
        st = [self._sr._stats([E[i] for i in idx], [R[i] for i in idx]) for idx in folds]
        total = [sum(x) for x in zip(*st)]
        models = [StreamRidge._solve([a - b for a, b in zip(total, s)]) for s in st]
        sse = np.zeros((len(self.alphas), K))
        sse_off = np.zeros(K)
        for i in range(n):
            s = models[fold_of[i]]
            t = R[i].T.astype(np.float64)
            Z = self._sr._design(s, E[i]) @ s["Q"]
            for ai, a in enumerate(self.alphas):
                p = Z @ (s["QB"] / (s["lam"] + a * s["scale"])[:, None]) + s["ym"]
                sse[ai] += ((p - t) ** 2).sum(0)
            sse_off += ((s["ym"] - t) ** 2).sum(0)
        best = np.argmin(sse, 0)
        self.gate = sse[best, np.arange(K)] < sse_off * (1 - self.margin)
        full = StreamRidge._solve(total)
        W = np.zeros((len(full["mu"]), K))
        for ai, a in enumerate(self.alphas):
            cols = np.where((best == ai) & self.gate)[0]
            if len(cols):
                W[:, cols] = full["Q"] @ (full["QB"][:, cols] / (full["lam"] + a * full["scale"])[:, None])
        self.s, self.W = full, W
        return self

    def predict(self, E):
        if not self.gate.any():
            return [np.zeros((len(self.gate), e.shape[1]), np.float32) for e in E]
        return [(self._sr._design(self.s, e) @ self.W + self.s["ym"] * self.gate).T.astype(np.float32) for e in E]


def gated(base, Ec, Yc, Et, feat):
    m = GatedResidualRidge(feat).fit(Ec, [y - o for y, o in zip(Yc, base.oof)])
    return [p + q for p, q in zip(base.predict(Et), m.predict(Et))], float(m.gate.mean())


# ----------------------------------------------------------------------------- per subject
def analyze(E, Y, a, log=print):
    n = len(E)
    _, test = chrono_split(n, 0.2, gap=1)
    Et, Yt = [E[i] for i in test], [Y[i] for i in test]
    rfeat = RandomConvFeatures(E[0].shape[0], M=a.n_random, seed=a.seed)
    wfeat = MorletEnvBank()
    res = {"n_trials": n, "fracs": {}}
    for f in a.fracs:
        n_c = max(3, int(round(n * f)))
        if n_c >= test[0]:
            continue
        Ec, Yc = E[:n_c], Y[:n_c]
        bands = abs(f - 0.2) < 1e-9
        out, on = {}, {}

        def score(name, P):
            m = per_contact_metrics(P, Yt, FS, bands=bands)
            out[name] = {"mean": {k: float(np.nanmean(v)) for k, v in m.items()}, "r": [float(x) for x in m["r"]]}

        score("ridge", StreamRidge(LAG9).fit(Ec, Yc).predict(Et))
        base = KFoldRidge(LAG9, shared=True).fit(Ec, Yc)
        score("base", base.predict(Et))
        P, on["env_gated"] = gated(base, Ec, Yc, Et, feat_env)
        score("env_gated", P)
        P, on["wav_gated"] = gated(base, Ec, Yc, Et, wfeat)
        score("wav_gated", P)
        P, on["rocket_gated"] = gated(base, Ec, Yc, Et, rfeat)
        score("rocket_gated", P)
        if not a.no_all:
            P, on["all_gated"] = gated(base, Ec, Yc, Et, concat_feats(wfeat, rfeat))
            score("all_gated", P)
        mins = sum(e.shape[1] for e in Ec) / FS / 60
        res["fracs"][f"{f:g}"] = {"n_calib": n_c, "minutes": mins, "gate_on": on,
                                  "mean": {k: v["mean"]["r"] for k, v in out.items()},
                                  "bands": {k: v["mean"] for k, v in out.items()} if bands else {},
                                  "r": {k: v["r"] for k, v in out.items()}}
        log(f"    f={f:g} ({mins:.1f} min) " + " ".join(f"{k}={v['mean']['r']:+.3f}" for k, v in out.items()) +
            " | gate on: " + " ".join(f"{k}={v:.2f}" for k, v in on.items()))
    return res


def summarize(results, out_dir, log=print):
    from scipy.stats import wilcoxon

    def wp(x, y):
        try:
            return float(wilcoxon(x, y).pvalue)
        except ValueError:
            return float("nan")

    lines, summ = [], {}
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r for r in results if f in r["fracs"]]
        models = list(rows[0]["fracs"][f]["mean"])
        M = {m: np.array([r["fracs"][f]["mean"][m] for r in rows]) for m in models}
        mins = np.mean([r["fracs"][f]["minutes"] for r in rows])
        lines += [f"\n### calib fraction {f}  ({len(rows)} subjects, mean {mins:.1f} min)",
                  "| model | mean r | vs ridge | p | vs base | wins vs base | p | gate on |",
                  "|---|---|---|---|---|---|---|---|"]
        summ[f] = {}
        for m in models:
            d0, d1 = M[m] - M["ridge"], M[m] - M["base"]
            gon = np.mean([r["fracs"][f]["gate_on"].get(m, np.nan) for r in rows])
            p0 = wp(M[m], M["ridge"]) if m != "ridge" else float("nan")
            p1 = wp(M[m], M["base"]) if m not in ("ridge", "base") else float("nan")
            summ[f][m] = {"mean": float(M[m].mean()), "diff_ridge": float(d0.mean()), "p_ridge": p0,
                          "diff_base": float(d1.mean()), "wins_base": int((d1 > 0).sum()), "p_base": p1,
                          "gate_on": float(gon)}
            lines.append(f"| {m} | {M[m].mean():.3f} | {d0.mean():+.4f} | {p0:.4f} | {d1.mean():+.4f} | "
                         f"{(d1 > 0).sum()}/{len(d1)} | {p1:.4f} | {gon:.2f} |")
    if "0.2" in fracs:
        rows = [r for r in results if "0.2" in r["fracs"]]
        models = list(rows[0]["fracs"]["0.2"]["bands"])
        keys = ["r", "r2", "r_delta", "r_theta", "r_alpha", "r_beta", "env_alpha", "env_beta", "env_gamma"]
        keys += [k for k in ("r_no50", "r_induced") if k in rows[0]["fracs"]["0.2"]["bands"][models[0]]]
        lines += ["\n### calib fraction 0.2: by band", "| model | " + " | ".join(keys) + " |",
                  "|---" * (len(keys) + 1) + "|"]
        for m in models:
            lines.append(f"| {m} | " + " | ".join(
                f"{np.nanmean([r['fracs']['0.2']['bands'][m].get(k, np.nan) for r in rows]):.3f}" for k in keys) + " |")
        models = list(rows[0]["fracs"]["0.2"]["mean"])
        lines += ["\n### per subject (calib fraction 0.2)", "| subject | " + " | ".join(models) + " |",
                  "|---" * (len(models) + 1) + "|"]
        for r in rows:
            lines.append(f"| {r['subject']} | " + " | ".join(f"{r['fracs']['0.2']['mean'][m]:.3f}" for m in models) + " |")
    md = "\n".join(lines)
    log(md)
    if out_dir:
        with open(os.path.join(out_dir, "summary.md"), "w", encoding="utf-8") as fh:
            fh.write(md + "\n")
        with open(os.path.join(out_dir, "summary.json"), "w") as fh:
            json.dump(summ, fh, indent=1)
    return summ


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--min_trials", type=int, default=20)
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.05, 0.1, 0.2])
    ap.add_argument("--n_random", type=int, default=512)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no_all", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "nonlin.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    from s2i.ceiling import load_trials
    log(f"config: {vars(a)}")
    results = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid = os.path.basename(p)[:-3]
        fp = os.path.join(a.out, f"nonlin_{sid}.json")
        if os.path.exists(fp) and not a.overwrite:
            results.append(json.load(open(fp)))
            log(f"[{sid}] already done, loaded")
            continue
        t0 = time.time()
        s, E, Y = load_trials(p, "bip", "car")
        if len(E) < a.min_trials or s.n_targets == 0:
            log(f"[{sid}] skipped")
            continue
        log(f"[{sid}] {len(E)} trials, {len(s.eeg_names)} scalp, {s.n_targets} targets")
        r = analyze(E, Y, a, log)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {time.time() - t0:.0f}s")
    if results:
        summarize(results, a.out, log)


if __name__ == "__main__":
    main()

## 2b. Kiểm tra độ nhạy trên dữ liệu giả (khoảng 2 phút)
Mô phỏng một subject có thành phần tuyến tính, cộng thêm (a) envelope alpha, (b) envelope gamma 35–50 Hz, hoặc (c) không có gì phi tuyến. Kết quả kỳ vọng:
- **(a)**: `env_gated` và `wav_gated` > `base`.
- **(b)**: `wav_gated` > `base`, còn `env_gated` ≈ `base` vì không có dải gamma.
- **(c)**: mọi cổng đóng (`gate on` = 0), tất cả bằng `base`.

Bước này đảm bảo rằng nếu kết quả thật ra âm tính thì đó là do dữ liệu, không phải do probe bị mù.

In [ ]:
#@title 2b. Smoke test độ nhạy
import importlib, types, numpy as np
from scipy.signal import butter, sosfiltfilt, hilbert
import s2i.nonlin as NL
importlib.reload(NL)

def fake_subject(band, seed=2, n=80, C=19, K=12):
    g = np.random.default_rng(seed); E, Y = [], []
    A, B = g.normal(size=(K, C)), g.normal(size=(K, C))
    sos = butter(4, band, btype='band', fs=200, output='sos') if band else None
    for _ in range(n):
        L = int(g.integers(1400, 1800))
        e = g.normal(size=(C, L)).astype(np.float32)
        y = 0.3 * A @ e + 2 * g.normal(size=(K, L))
        if band:
            env = np.abs(hilbert(sosfiltfilt(sos, e, axis=1), axis=1))
            y += 0.5 * B @ (env - env.mean(1, keepdims=True))
        E.append(e); Y.append(y.astype(np.float32))
    return E, Y

args = types.SimpleNamespace(fracs=[0.2], n_random=512, seed=0, no_all=True)
for name, band in [('(a) alpha envelope', (8, 13)), ('(b) gamma envelope', (35, 50)), ('(c) không phi tuyến', None)]:
    print(name)
    NL.analyze(*fake_subject(band), args)

## 3. Chạy trên dữ liệu thật
Mỗi subject mất khoảng 2–6 phút, tổng cộng khoảng 1 giờ cho 15 subject. Log được ghi vào `runs/nonlin_bip_car/nonlin.log`.
Nếu Colab bị ngắt kết nối: chạy lại cell 1 rồi cell 3, các subject đã xong sẽ được bỏ qua.

In [ ]:
#@title 3. Chạy giai đoạn 1c
!python -m s2i.nonlin --data {DATA_DIR} --out {OUT_DIR}

In [ ]:
#@title 4. Kết quả
import json, numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT_DIR}/summary.md', encoding='utf-8').read()))

S = json.load(open(f'{OUT_DIR}/summary.json'))
fracs = sorted(S, key=float)
models = [m for m in S[fracs[0]] if m not in ('ridge', 'base')]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(fracs)); w = 0.8 / len(models)
for i, m in enumerate(models):
    ax[0].bar(x + i * w, [S[f][m]['diff_base'] for f in fracs], w, label=m)
    ax[1].bar(x + i * w, [S[f][m]['gate_on'] for f in fracs], w, label=m)
for a_, t in zip(ax, ['mean r − base', 'tỉ lệ contact bật cổng']):
    a_.set_xticks(x + w * (len(models) - 1) / 2); a_.set_xticklabels([f'{float(f):.0%} calib' for f in fracs])
    a_.set_title(t); a_.axhline(0, color='k', lw=0.8); a_.grid(alpha=0.3, axis='y')
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/nonlin.png', dpi=130); plt.show()

## 5. Cách đọc kết quả và quyết định bước tiếp theo
Tiêu chí "thắng": hơn `base` với Wilcoxon p < 0,05 **và** thắng ở ≥ 12/15 subject, ở mức calib 20%. Đồng thời R² không được giảm so với `base` (xem bảng theo band).

| Kết quả | Kết luận | Thiết kế encoder–decoder ở giai đoạn 2 (GPU) |
|---|---|---|
| `wav_gated` thắng, `rocket_gated` thì không | Thông tin phi tuyến chủ yếu là **công suất phổ theo kênh** | Front-end filter-bank/wavelet cố định hoặc bán cố định, mỗi kênh một token, readout ridge K-fold theo từng subject. Encoder nhỏ, pretrain tự giám sát trên EEG |
| `rocket_gated` hoặc `all_gated` thắng rõ hơn `wav_gated` | Có **tương tác giữa các kênh** hoặc dạng sóng phi tuyến mà phổ theo kênh không bắt được | Encoder tích chập/attention học được trên không gian kênh, đáng đầu tư GPU (pretrain + readout K-fold) |
| Chỉ thắng ở alpha/beta (bảng theo band), r tổng tăng ít | Thông tin thật nhưng nhỏ | Giữ nhánh tuyến tính (delta/theta) + nhánh envelope có cổng. Deep chỉ đóng vai trò bộ trích đặc trưng |
| Không model nào thắng, `gate on` < 10% | Với 1–4 phút calib, đặc trưng phi tuyến tổng quát không giúp | SOTA trong protocol hiện tại = `base`. Muốn deep thắng thì phải có **thêm dữ liệu**: đoạn giữa các trial, calib rải theo session, hoặc pretrain trên EEG ngoài |

Các cột cần xem thêm:
- **`gate on`**: tỉ lệ contact bật tầng 2. Cao mà r không tăng nghĩa là CV đang lạc quan.
- **So `env_gated` với `wav_gated` ở `r_delta`**: tầng 2 không được làm hỏng delta (lỗi của 1b).
- **Theo subject**: subject ít trial (sub-13) không được bị phạt nặng như ở 1b.

Gửi lại bảng ở cell 4 (hoặc chỉ cần lưu notebook) để quyết định thiết kế giai đoạn 2.